# Debugging code
## Week 4 - Discussion section

### Load the data and agree on an expectation

In [1]:
import io
import pandas as pd
import numpy as np

CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

stream = pd.read_csv(io.StringIO(CSV))

In [2]:
stream

,site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
0,SB-01,2024-01-15,12.4,"1,240",0.82,good
1,SB-01,2024-02-15,13.1,"1,180",0.79,good
2,SB-01,2024-03-15,15.8,"2,050",1.21,good
3,SB-01,2024-04-15,18.2,"1,620",0.94,good
4,sb-01,2024-05-15,21.5,980,0.66,good
5,SB-01,2024-06-15,-999.0,540,0.51,suspect
6,SB-02,2024-01-15,11.8,"3,410",1.88,good
7,SB-02,2024-02-15,12.3,"3,290",1.92,good
8,SB-02,2024-03-15,14.9,"4,870",2.41,good
9,SB-02,2024-04-15,17.6,"3,950",2.09,estimated


How many monitoring sites are in this dataset?
- Three

What is a plausible range for stream water temperature in degrees Celsius?
- 10.2 to 21.5

Roughly what mean water temperature do you expect across all measurements?
- About 15.5

### A worked example 

In [ ]:
# Step 1: read the error
stream['discharge_cfs'] * 0.0283168

In [3]:
# Step 2: look at the data
stream.dtypes

site_id           object
date              object
water_temp_c     float64
discharge_cfs     object
nitrate_mg_l      object
qc_flag           object
dtype: object

In [ ]:
# Step 3: isolate 
stream['discharge_cfs'].iloc[0] * 0.0283168   # same TypeError, one value
stream['discharge_cfs'].iloc[0]              # '1,240'

In [4]:
# Step 4: form a hypothesis and test it
stream['discharge_cfs'].unique()

array(['1,240', '1,180', '2,050', '1,620', '980', '540', '3,410', '3,290',
       '4,870', '3,950', '2,240', '1,130', '620', '590', '1,040', '870',
       '410', '180'], dtype=object)

In [5]:
# Step 5: fix and verify 
discharge_cms = (stream['discharge_cfs'].str.replace(',','', regex=False).astype(float)) * 0.0283168

discharge_cms.head(3)

0    35.112832
1    33.413824
2    58.049440
Name: discharge_cfs, dtype: float64

In [6]:
# Step 6: consider adding a guard 
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

In [7]:
# If you can't fix it, create a minimal reproducible example (MRE) and ask for help.

import pandas as pd
df = pd.DataFrame({'discharge_cfs': ['1,240', '980']})
df['discharge_cfs'] * 0.0283168     # TypeError: can't multiply sequence by non-int

TypeError: can't multiply sequence by non-int of type 'float'


| What I expected | What happened | My hypothesis | What I changed | Result |
| -------- | -------- | -------- | -------- | -------- |
| Discharge in m³/s, around 35   | TypeError: can't multiply sequence by non-int   | discharge_cfs is text because of thousands separators   | .str.replace(',', '') then .astype(float) | Works, 35.11 m³/s, guard added |


### On your own

In [8]:
stream['site_id'] = stream['site_id'].str.strip()

print(stream['site_id'].unique())

# prediction before running: this doesn't fully help get us to the three sites 
# as the strings are mixed upper and lower case 

# ran it, got AttributeError: 'Series' object has no attribute 'strip' - maybe needs str.strip? 

# did .str.strip, but didn't get rid of spaces - try making it equal the stream['site_id'] column 

# that worked! 

['SB-01' 'sb-01' 'SB-02' 'sb-02' 'SB-03']


In [9]:
stream['nitrate_mg_l'].astype(float)

# prediction: this won't work because of symbols <

# correct. decided as a team to convert those values 0.05.

ValueError: could not convert string to float: '<0.05'

In [10]:
# strip '<0.05' to 0.05
stream['nitrate_mg_l'] = stream['nitrate_mg_l'].str.replace('<', '')


In [11]:
# check that it worked by looking at the df
stream 

# it worked! 

,site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
0,SB-01,2024-01-15,12.4,"1,240",0.82,good
1,SB-01,2024-02-15,13.1,"1,180",0.79,good
2,SB-01,2024-03-15,15.8,"2,050",1.21,good
3,SB-01,2024-04-15,18.2,"1,620",0.94,good
4,sb-01,2024-05-15,21.5,980,0.66,good
5,SB-01,2024-06-15,-999.0,540,0.51,suspect
6,SB-02,2024-01-15,11.8,"3,410",1.88,good
7,SB-02,2024-02-15,12.3,"3,290",1.92,good
8,SB-02,2024-03-15,14.9,"4,870",2.41,good
9,SB-02,2024-04-15,17.6,"3,950",2.09,estimated


In [12]:
# stream['temperature'].mean()

# prediction: this will not work because the column name is wrong
# replace with 'water_temp_c'

stream['water_temp_c'].mean()
# that worked! but it gave a wrong number! more on this below 



-153.8111111111111

In [13]:
# stream.iloc[18] 
# prediction: this doesn't work because index starts at 0. to see the last row, 
# run 17 in the brackets

stream.iloc[17] 
# that worked! 


site_id               SB-03
date             2024-06-15
water_temp_c         -999.0
discharge_cfs           180
nitrate_mg_l           0.05
qc_flag             suspect
Name: 17, dtype: object

### Bugs that do not raise errors 

In [ ]:
stream['water_temp_c'].mean()

# prediction: this will give you an output but it will be wrong because 
# temp values include NAs of -999.0

-153.8111111111111

In [ ]:
# to resolve: list -999.0 as NaN and remove NAs when calculating the mean 
stream['water_temp_c'].replace[-999.0, np.nan]

# lab ended, figure out what is happening here... 

TypeError: 'method' object is not subscriptable